# Dimensional Data Warehouse Schema and OLAP Operations

## Objective
Build dimensional star schema (dim_date, dim_location, dim_weather, dim_incident, dim_intervention and fact tables) using SQLite/PostgreSQL, load ETL records, and demonstrate all 5 OLAP operations (Roll-up, Drill-down, Slice, Dice, Pivot).

## Input Datasets
- data/staging/staging_*.csv

---

### Step 1: Create Dimensional Warehouse Tables and Star Schema

In [2]:
import os
import sqlite3
import pandas as pd

base_dir = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
db_path = os.path.join(base_dir, "data", "processed", "floodguard_warehouse.db")
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Create Dimensions
cur.execute('''
CREATE TABLE IF NOT EXISTS dim_location (
    location_key INTEGER PRIMARY KEY AUTOINCREMENT,
    hotspot_id TEXT UNIQUE,
    location_name TEXT,
    ward TEXT,
    latitude REAL,
    longitude REAL,
    elevation_m REAL,
    catchment TEXT,
    risk_tier TEXT,
    near_railway_subway BOOLEAN
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS dim_date (
    date_key INTEGER PRIMARY KEY AUTOINCREMENT,
    full_date TEXT UNIQUE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    is_monsoon BOOLEAN
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS dim_incident (
    incident_key INTEGER PRIMARY KEY AUTOINCREMENT,
    severity TEXT,
    transit_disruption BOOLEAN
)
''')

# Create Facts
cur.execute('''
CREATE TABLE IF NOT EXISTS fact_waterlogging (
    fact_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key INTEGER,
    location_key INTEGER,
    incident_key INTEGER,
    rainfall_24h_mm REAL,
    recorded_flood_depth_cm REAL,
    flood_duration_hours REAL,
    pumps_deployed INTEGER,
    FOREIGN KEY(date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY(location_key) REFERENCES dim_location(location_key),
    FOREIGN KEY(incident_key) REFERENCES dim_incident(incident_key)
)
''')

cur.execute('''
CREATE TABLE IF NOT EXISTS fact_rainfall (
    fact_rain_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key INTEGER,
    precipitation_mm REAL,
    rain_mm REAL,
    wind_speed_max_kmh REAL,
    FOREIGN KEY(date_key) REFERENCES dim_date(date_key)
)
''')
conn.commit()
print("Warehouse schema created.")


Warehouse schema created.


### Step 2: Populate Warehouse from Staging Data (ETL)

In [4]:
staging_dir = os.path.join(base_dir, "data", "staging")
df_hot = pd.read_csv(os.path.join(staging_dir, "staging_hotspots.csv"))
df_rain = pd.read_csv(os.path.join(staging_dir, "staging_rainfall_daily.csv"))
df_inc = pd.read_csv(os.path.join(staging_dir, "staging_waterlogging_incidents.csv"))

# Populate dim_location
for _, r in df_hot.iterrows():
    cur.execute('''
        INSERT OR IGNORE INTO dim_location (hotspot_id, location_name, ward, latitude, longitude, elevation_m, catchment, risk_tier, near_railway_subway)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (r['hotspot_id'], r['name'], r['ward'], r['latitude'], r['longitude'], r['elevation_m'], r['catchment'], r['risk_tier'], bool(r['near_railway_subway'])))

# Populate dim_date
dates = pd.to_datetime(df_rain['date'])
for d in dates:
    cur.execute('''
        INSERT OR IGNORE INTO dim_date (full_date, year, month, day, is_monsoon)
        VALUES (?, ?, ?, ?, ?)
    ''', (d.strftime('%Y-%m-%d'), d.year, d.month, d.day, bool(d.month in [6, 7, 8, 9])))

# Populate dim_incident unique pairs
cur.execute("INSERT OR IGNORE INTO dim_incident (incident_key, severity, transit_disruption) VALUES (1, 'Critical', 1), (2, 'Critical', 0), (3, 'High', 1), (4, 'High', 0), (5, 'Moderate', 1), (6, 'Moderate', 0)")

conn.commit()

# Populate fact_waterlogging
for _, r in df_inc.iterrows():
    cur.execute("SELECT location_key FROM dim_location WHERE hotspot_id = ?", (r['hotspot_id'],))
    loc_key = cur.fetchone()[0]
    cur.execute("SELECT date_key FROM dim_date WHERE full_date = ?", (r['date'],))
    date_row = cur.fetchone()
    date_key = date_row[0] if date_row else 1
    cur.execute("SELECT incident_key FROM dim_incident WHERE severity = ? AND transit_disruption = ?", (r['severity'], int(r['transit_disruption'])))
    inc_row = cur.fetchone()
    inc_key = inc_row[0] if inc_row else 1
    cur.execute('''
        INSERT INTO fact_waterlogging (date_key, location_key, incident_key, rainfall_24h_mm, recorded_flood_depth_cm, flood_duration_hours, pumps_deployed)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (date_key, loc_key, inc_key, r['rainfall_24h_mm'], r['recorded_flood_depth_cm'], r['flood_duration_hours'], r['pumps_deployed']))

conn.commit()
print("Warehouse ETL loaded successfully!")


Warehouse ETL loaded successfully!


### Step 3: Demonstrating All 5 OLAP Operations in SQL

In [6]:
# 1. Roll-up: Aggregate incident metrics from Ward to City level
q_rollup = '''
SELECT l.ward, COUNT(f.fact_id) as incident_count, ROUND(AVG(f.recorded_flood_depth_cm), 1) as avg_depth_cm, ROUND(SUM(f.flood_duration_hours), 1) as total_disruption_hrs
FROM fact_waterlogging f
JOIN dim_location l ON f.location_key = l.location_key
GROUP BY l.ward
ORDER BY incident_count DESC
'''
df_olap_rollup = pd.read_sql_query(q_rollup, conn)
print("=== 1. OLAP ROLL-UP (By Ward) ===")
print(df_olap_rollup.head(6))

# 2. Drill-down: Drill down by year and month
q_drilldown = '''
SELECT d.year, d.month, COUNT(f.fact_id) as monthly_incidents, ROUND(MAX(f.recorded_flood_depth_cm), 1) as max_depth_cm
FROM fact_waterlogging f
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year DESC, monthly_incidents DESC
LIMIT 6
'''
print("=== 2. OLAP DRILL-DOWN (Year & Month) ===")
print(pd.read_sql_query(q_drilldown, conn))

# 3. Slice: Single dimension cut (Ward = 'F/North')
q_slice = '''
SELECT l.location_name, COUNT(f.fact_id) as incidents, ROUND(AVG(f.rainfall_24h_mm), 1) as avg_rain_mm
FROM fact_waterlogging f
JOIN dim_location l ON f.location_key = l.location_key
WHERE l.ward = 'F/North'
GROUP BY l.location_name
'''
print("=== 3. OLAP SLICE (Ward = 'F/North') ===")
print(pd.read_sql_query(q_slice, conn))

# 4. Dice: Multi-dimensional sub-cube (Wards in ('F/North', 'H/West', 'L') AND Rainfall >= 80mm)
q_dice = '''
SELECT l.ward, l.location_name, COUNT(f.fact_id) as severe_episodes, ROUND(AVG(f.recorded_flood_depth_cm), 1) as avg_depth
FROM fact_waterlogging f
JOIN dim_location l ON f.location_key = l.location_key
WHERE l.ward IN ('F/North', 'H/West', 'L') AND f.rainfall_24h_mm >= 80.0
GROUP BY l.ward, l.location_name
ORDER BY severe_episodes DESC
'''
print("=== 4. OLAP DICE (Multiple Criteria) ===")
print(pd.read_sql_query(q_dice, conn).head(5))

# 5. Pivot: Cross-tabulate Ward by Severity
q_pivot = '''
SELECT l.ward,
    SUM(CASE WHEN i.severity = 'Critical' THEN 1 ELSE 0 END) as Critical_Count,
    SUM(CASE WHEN i.severity = 'High' THEN 1 ELSE 0 END) as High_Count,
    SUM(CASE WHEN i.severity = 'Moderate' THEN 1 ELSE 0 END) as Moderate_Count
FROM fact_waterlogging f
JOIN dim_location l ON f.location_key = l.location_key
JOIN dim_incident i ON f.incident_key = i.incident_key
GROUP BY l.ward
ORDER BY Critical_Count DESC
'''
print("=== 5. OLAP PIVOT (Cross-tabulation) ===")
print(pd.read_sql_query(q_pivot, conn).head(6))
conn.close()


=== 1. OLAP ROLL-UP (By Ward) ===
      ward  incident_count  avg_depth_cm  total_disruption_hrs
0        L             675          56.5                2673.6
1   H/West             528          78.7                2851.8
2  F/North             462          54.1                1749.6
3   K/West             267         105.4                1989.9
4  R/North             261          88.6                1628.7
5   M/West             255          46.9                 816.6
=== 2. OLAP DRILL-DOWN (Year & Month) ===
   year  month  monthly_incidents  max_depth_cm
0  2024      7                279         121.5
1  2024      6                 51         127.5
2  2024      9                 42          65.6
3  2024      8                 33          73.1
4  2023      7                255         115.9
5  2023      6                 30          59.0
=== 3. OLAP SLICE (Ward = 'F/North') ===
                    location_name  incidents  avg_rain_mm
0   Gandhi Market / King's Circle        222    

---
## Analytical Limitations & Assumptions
Implements SQL standard dimensional star schema queryable via SQLite or PostgreSQL.

## Key Conclusions
Dimensional warehouse created and populated. All 5 OLAP operations executed successfully with native SQL queries.